# 14: Sharing through WhatsApp (E6, part 2)

Notebook 13 exported 100 original files (50 real photographs, 50 images from five current generators) to `whatsapp/original/`. These were sent through WhatsApp as ordinary photos (standard quality), and the received files were exported back to Drive as `whatsapp/received.zip`.

WhatsApp renames, shrinks and re-compresses every photo. This notebook matches each received file to its original by comparing small thumbnails, then runs the four detectors on both versions with the same 224 x 224 centre crop.

GPU runtime recommended (CPU works but is slower).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import io
import json
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from torchvision.models import resnet50
from transformers import CLIPVisionModelWithProjection

ROOT = Path("/content/drive/MyDrive/minor-project")
RESULTS = ROOT / "results"
WHATSAPP = ROOT / "whatsapp"
RECEIVED = Path("/content/received")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Settings

In [ ]:
SIZE = 224
CLIP_NAME = "openai/clip-vit-large-patch14"
BATCH_SIZE = 64
COLORS = {"M1 ResNet-50": "#2a78d6", "M2 CLIP": "#eb6834", "M3 NPR": "#1baf7a", "M4 M2+M3": "#eda100"}

## Helper functions (same as notebook 03)

In [ ]:
STD_LUMA = np.array([
    16, 11, 10, 16, 24, 40, 51, 61,
    12, 12, 14, 19, 26, 58, 60, 55,
    14, 13, 16, 24, 40, 57, 69, 56,
    14, 17, 22, 29, 51, 87, 80, 62,
    18, 22, 37, 56, 68, 109, 103, 77,
    24, 35, 55, 64, 81, 104, 113, 92,
    49, 64, 78, 87, 103, 121, 120, 101,
    72, 92, 95, 98, 112, 100, 103, 99,
])


def estimate_quality(img):
    luma = np.array(img.quantization[0])
    best_quality, best_error = 0, np.inf
    for quality in range(1, 101):
        scale = 5000 // quality if quality < 50 else 200 - 2 * quality
        table = np.clip((STD_LUMA * scale + 50) // 100, 1, 255)
        error = np.abs(table - luma).sum()
        if error < best_error:
            best_quality, best_error = quality, error
    return best_quality


def center_crop_aligned(img, size=SIZE):
    width, height = img.size
    left = ((width - size) // 2) // 16 * 16
    top = ((height - size) // 2) // 16 * 16
    return img.crop((left, top, left + size, top + size))

## Unpack the received files

Accepts `whatsapp/received.zip` (a WhatsApp chat export) or a folder `whatsapp/received/`. Non-image files such as the chat text are ignored.

In [ ]:
RECEIVED.mkdir(exist_ok=True)
if (WHATSAPP / "received.zip").exists():
    with zipfile.ZipFile(WHATSAPP / "received.zip") as z:
        z.extractall(RECEIVED)
    source_dir = RECEIVED
else:
    source_dir = WHATSAPP / "received"

received_files = [p for p in source_dir.rglob("*")
                  if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp") and not p.name.startswith(".")]
originals = pd.read_csv(WHATSAPP / "original_meta.csv")
print(f"{len(received_files)} received images, {len(originals)} originals")

## Match each received file to its original

Both versions are shrunk to 32 x 32 grey thumbnails; the pairing with the smallest total difference is chosen (each original used once). Pairs whose aspect ratio differs, or whose thumbnails still differ a lot, are reported and left out.

In [ ]:
def thumbnail(img):
    return np.asarray(img.convert("L").resize((32, 32), Image.BILINEAR), dtype=np.float32) / 255


orig_imgs = [Image.open(WHATSAPP / "original" / f) for f in originals["file"]]
recv_imgs = [Image.open(p) for p in received_files]
orig_thumbs = np.stack([thumbnail(i) for i in orig_imgs]).reshape(len(orig_imgs), -1)
recv_thumbs = np.stack([thumbnail(i) for i in recv_imgs]).reshape(len(recv_imgs), -1)
orig_aspect = np.array([i.size[0] / i.size[1] for i in orig_imgs])
recv_aspect = np.array([i.size[0] / i.size[1] for i in recv_imgs])

cost = np.abs(recv_thumbs[:, None, :] - orig_thumbs[None, :, :]).mean(-1)
cost += 10 * (np.abs(np.log(recv_aspect[:, None] / orig_aspect[None, :])) > 0.02)
r_idx, o_idx = linear_sum_assignment(cost)
good = cost[r_idx, o_idx] < 0.05
print(f"Matched {good.sum()} pairs; rejected {(~good).sum()}")

pairs = originals.iloc[o_idx[good]].reset_index(drop=True).copy()
pairs["received_file"] = [received_files[i].name for i in r_idx[good]]
pairs["match_error"] = cost[r_idx, o_idx][good]
pairs["received_width"] = [recv_imgs[i].size[0] for i in r_idx[good]]
pairs["received_height"] = [recv_imgs[i].size[1] for i in r_idx[good]]
pairs["received_quality"] = [estimate_quality(recv_imgs[i]) if recv_imgs[i].format == "JPEG" else np.nan for i in r_idx[good]]
pairs["y"] = (pairs["label"] == "fake").astype(int)
display(pairs.groupby("source")[["width", "received_width", "received_quality"]].median())

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6.6))
for col, k in enumerate([0, len(pairs) // 3, 2 * len(pairs) // 3, len(pairs) - 1]):
    row = pairs.iloc[k]
    a = Image.open(WHATSAPP / "original" / row["file"])
    b = recv_imgs[r_idx[good][k]]
    for r, (img, label) in enumerate([(a, "original"), (b, "after WhatsApp")]):
        axes[r, col].imshow(img.convert("RGB"))
        axes[r, col].set_title(f"{row['source']}, {label}\n{img.size[0]} x {img.size[1]}", fontsize=9)
        axes[r, col].axis("off")
plt.tight_layout()
plt.show()

## Load the detectors (as in notebook 11)

In [ ]:
class NPRNet(nn.Module):
    def __init__(self):
        super().__init__()
        base = resnet50(weights=None)
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1, bias=False),
            base.bn1, base.relu, base.maxpool,
        )
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, 1)

    def forward(self, x):
        rebuilt = F.interpolate(F.interpolate(x, scale_factor=0.5, mode="nearest"), scale_factor=2, mode="nearest")
        npr = x - rebuilt
        x = self.stem(npr * 2.0 / 3.0)
        x = self.layer2(self.layer1(x))
        return self.fc(torch.flatten(self.pool(x), 1))


m1 = resnet50(weights=None)
m1.fc = nn.Linear(m1.fc.in_features, 1)
m1.load_state_dict(torch.load(ROOT / "checkpoints" / "m1_resnet50.pt", map_location=device))
m1 = m1.to(device).eval()

m3 = NPRNet()
m3.load_state_dict(torch.load(ROOT / "checkpoints" / "m3_npr.pt", map_location=device))
m3 = m3.to(device).eval()

dtype = torch.float16 if device.type == "cuda" else torch.float32
clip = CLIPVisionModelWithProjection.from_pretrained(CLIP_NAME, torch_dtype=dtype).to(device).eval()
CLIP_SIZE = clip.config.image_size

best_C = json.load(open(RESULTS / "m2_clip.json"))["best_C"]
train_feats = np.load(ROOT / "features" / f"{CLIP_NAME.split('/')[-1]}_train.npy")
train_y = (pd.read_csv(ROOT / "processed" / "train_meta.csv")["label"] == "fake").astype(int).to_numpy()
clf = LogisticRegression(C=best_C, max_iter=5000).fit(train_feats, train_y)

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


@torch.no_grad()
def score_all(images):
    probs = {"M1 ResNet-50": [], "M2 CLIP": [], "M3 NPR": []}
    for start in range(0, len(images), BATCH_SIZE):
        x = torch.from_numpy(np.array(images[start:start + BATCH_SIZE])).to(device).permute(0, 3, 1, 2).float() / 255
        with torch.autocast(device.type, enabled=device.type == "cuda"):
            imagenet_x = (x - IMAGENET_MEAN) / IMAGENET_STD
            probs["M1 ResNet-50"].append(torch.sigmoid(m1(imagenet_x).squeeze(1).float()).cpu().numpy())
            probs["M3 NPR"].append(torch.sigmoid(m3(imagenet_x).squeeze(1).float()).cpu().numpy())
        clip_x = x if x.shape[-1] == CLIP_SIZE else F.interpolate(x, size=CLIP_SIZE, mode="bicubic", align_corners=False)
        emb = clip(pixel_values=((clip_x - CLIP_MEAN) / CLIP_STD).to(dtype)).image_embeds.float()
        probs["M2 CLIP"].append(F.normalize(emb, dim=1).cpu().numpy())
    probs = {k: np.concatenate(v) for k, v in probs.items()}
    probs["M2 CLIP"] = clf.predict_proba(probs["M2 CLIP"])[:, 1]
    probs["M4 M2+M3"] = (probs["M2 CLIP"] + probs["M3 NPR"]) / 2
    return probs


print("Detectors loaded")

## Score both versions

In [ ]:
def crops_of(images):
    return np.stack([np.asarray(center_crop_aligned(img.convert("RGB"))) for img in images])


before = crops_of([Image.open(WHATSAPP / "original" / f) for f in pairs["file"]])
after = crops_of([recv_imgs[i] for i in r_idx[good]])
p_before, p_after = score_all(before), score_all(after)

for model in COLORS:
    pairs[f"{model} before"] = p_before[model]
    pairs[f"{model} after"] = p_after[model]
pairs.to_csv(RESULTS / "e6_whatsapp_predictions.csv", index=False)

y = pairs["y"].to_numpy()
rows = []
for model in COLORS:
    for when, p in [("before", p_before[model]), ("after", p_after[model])]:
        rows.append({"model": model, "when": when,
                     "real_called_real": float((p[y == 0] < 0.5).mean()),
                     "fake_detected": float((p[y == 1] >= 0.5).mean()),
                     "auc": float(roc_auc_score(y, p))})
summary = pd.DataFrame(rows)
summary.to_csv(RESULTS / "e6_whatsapp_summary.csv", index=False)
display(summary.pivot(index="model", columns="when").reindex(columns=["before", "after"], level=1).round(3))

per_source = pairs.groupby("source").apply(lambda g: pd.Series({
    **{f"{m} before": ((g[f"{m} before"] >= 0.5) == g["y"]).mean() for m in COLORS},
    **{f"{m} after": ((g[f"{m} after"] >= 0.5) == g["y"]).mean() for m in COLORS}}), include_groups=False)
per_source.to_csv(RESULTS / "e6_whatsapp_per_source.csv")
print("\nCorrect rate per source:")
display(per_source.round(2))

## Chart

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharey=True)
x = np.arange(len(COLORS))
w = 0.38
for ax, (metric, title) in zip(axes, [("real_called_real", "Real photos called real (%)"),
                                      ("fake_detected", "AI images detected (%)"), ("auc", "ROC-AUC (x100)")]):
    b = summary[summary["when"] == "before"].set_index("model").loc[list(COLORS), metric] * 100
    a = summary[summary["when"] == "after"].set_index("model").loc[list(COLORS), metric] * 100
    bars1 = ax.bar(x - w / 2, b, w - 0.03, color="#a9b4c2", label="Original file")
    bars2 = ax.bar(x + w / 2, a, w - 0.03, color="#2a78d6", label="After WhatsApp")
    ax.bar_label(bars1, fmt="%.0f", fontsize=8, color="#555555")
    ax.bar_label(bars2, fmt="%.0f", fontsize=8)
    ax.set_xticks(x, [m.replace(" ", "\n", 1) for m in COLORS], fontsize=9)
    ax.set_title(title, fontsize=11)
    ax.axhline(50, color="#999999", linewidth=1, linestyle="--")
    ax.grid(axis="y", color="#e5e5e5")
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylim(0, 110)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False)
plt.tight_layout(rect=[0, 0, 1, 0.9])
plt.savefig(RESULTS / "e6_whatsapp.png", dpi=120)
plt.show()